In [51]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableConfig

from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END, MessagesState

from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import InMemorySaver # In memory checkpointing for graph -> short term memory
from langgraph.store.memory import InMemoryStore # Long term storage for graph -> long term memory

In [52]:
class AgentState(MessagesState):
    intent: str
    order_id: str
    preferred_language: str

In [53]:
class Context(TypedDict):
    user_id: str
    plan: str

In [54]:
def parse_request(state: AgentState, config: RunnableConfig)->dict:
    user_text=str(state["messages"][-1].content).strip()
    thread_id=config["configurable"]["thread_id"]

    print(f"[Node 1] thread_id: {thread_id}")
    print(f"[Node 1] message_count: {len(state["messages"])}")
    print(f"[Node 1] existing order: {state.get("order_id", "N/A")}")

    if user_text.startswith("My order ID is"):
        order_id=user_text.removeprefix("My order ID is").strip()
        return {
            "order_id": order_id,
            "intent": "save_order"
        }
    
    if user_text == "Where is my order?":
        return {
            "intent": "ask_order"
        }
    
    if user_text.startswith("Remember my language is"):
        return {
            "intent": "save_language"
        }
    
    if user_text == "Which language do I prefer":
        return {
            "intent": "ask_language"
        }
    
    return {
        "intent": "unknown"
    }

## Node for accessing Long Term memory store

In [55]:
def load_user_memory(state: AgentState, runtime: Runtime[Context])->dict:
    user_id=runtime.context.get("user_id")
    plan=runtime.context.get("plan")

    store=runtime.store

    namespace=("users", user_id)

    if state["intent"]=="save_language":
        message=str(state["messages"][-1].content).strip()
        language=message.removeprefix("Remember my language is").strip()
        store.put(
            namespace,
            "language",
            {"value": language}
        )
    
    memory=store.get(namespace, "language")

    language=(
        memory.value["value"]
        if memory is not None else "not set"
    )

    return {
        "preferred_language": language
    }

In [65]:
def respond(state: AgentState, config: RunnableConfig, runtime: Runtime[Context])->dict:
    thread_id=config["configurable"]["thread_id"]
    user_id=runtime.context.get("user_id")
    intent=state["intent"]

    if intent=="save_order":
        ans=(
            f"Saved order {state["order_id"]} for user {user_id} in thread {thread_id}"
        )
    
    elif intent=="ask_order":
        if state.get("order_id"):
            ans=(
                f"Your order {state["order_id"]} is being processed"
                "It will be shipped soon"
            )
        else:
            ans=(
                "Couldn't find your order. Please provide order ID"
            )
    
    elif intent=="save_language":
        ans=(
            f"Saved your preferred language {state["preferred_language"]}"
        )
    
    elif intent=="ask_language":
        if state.get("preferred_language"):
            ans=(
                f"Your preferred language is {state["preferred_language"]}"
            )
        else:
            ans=(
                "You haven't set your preferred language yet. "
                "Please se your preferred language."
            )
    
    else:
        ans=(
            "Ask about your order or language preferences."
        )
    
    return {
        "messages": [AIMessage(content=ans)]
    }

In [66]:
builder=StateGraph(state_schema=AgentState, context_schema=Context)

builder.add_node("parse_request", parse_request)
builder.add_node("load_user_memory", load_user_memory)
builder.add_node("respond", respond)

builder.add_edge(START, "parse_request")
builder.add_edge("parse_request", "load_user_memory")
builder.add_edge("load_user_memory", "respond")
builder.add_edge("respond", END)


checkpointer=InMemorySaver()
store=InMemoryStore()

graph=builder.compile(
    checkpointer=checkpointer,
    store=store
)

In [67]:
def chat(user_id: str, thread_id: str, message: str, plan: str="free"):
    
    print(f"\nUser {user_id} | Thread {thread_id}")
    print(f"\nInput {message}")

    result=graph.invoke({
        "messages": [HumanMessage(content=message)]
    },
    config={
        "configurable": {
            "thread_id": thread_id,
        },
        "tags": ["chat", 'demo']
    },
    context={
        "user_id": user_id,
        "plan": plan
    }
    )


    # The graph has been invoked
    print("Answer: ", result["messages"][-1].content)

    print("Total messages: ", len(result["messages"]))

    return result

In [68]:
chat("user_a", "a1", "My order ID is 1234567890", "premium")


User user_a | Thread a1

Input My order ID is 1234567890
[Node 1] thread_id: a1
[Node 1] message_count: 1
[Node 1] existing order: N/A
Answer:  Saved order 1234567890 for user user_a in thread a1
Total messages:  2


{'messages': [HumanMessage(content='My order ID is 1234567890', additional_kwargs={}, response_metadata={}, id='fd80b2aa-a139-4ba4-a29f-0a450c740aec'),
  AIMessage(content='Saved order 1234567890 for user user_a in thread a1', additional_kwargs={}, response_metadata={}, id='c9e3fe6e-0600-4fe9-9a47-d076af5b9917', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'save_order',
 'order_id': '1234567890',
 'preferred_language': 'not set'}

In [69]:
chat("user_a", "a1", "Where is my order?", "premium")


User user_a | Thread a1

Input Where is my order?
[Node 1] thread_id: a1
[Node 1] message_count: 3
[Node 1] existing order: 1234567890
Answer:  Your order 1234567890 is being processedIt will be shipped soon
Total messages:  4


{'messages': [HumanMessage(content='My order ID is 1234567890', additional_kwargs={}, response_metadata={}, id='fd80b2aa-a139-4ba4-a29f-0a450c740aec'),
  AIMessage(content='Saved order 1234567890 for user user_a in thread a1', additional_kwargs={}, response_metadata={}, id='c9e3fe6e-0600-4fe9-9a47-d076af5b9917', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='d79c903a-e6b7-49da-ab86-9785be4089ba'),
  AIMessage(content='Your order 1234567890 is being processedIt will be shipped soon', additional_kwargs={}, response_metadata={}, id='cfe6e4fc-7dc4-49d2-9212-927e74e27cc4', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'ask_order',
 'order_id': '1234567890',
 'preferred_language': 'not set'}

In [70]:
chat("user_a", "a1", "Remember my language is German", "premium")


User user_a | Thread a1

Input Remember my language is German
[Node 1] thread_id: a1
[Node 1] message_count: 5
[Node 1] existing order: 1234567890
Answer:  Saved your preferred language German
Total messages:  6


{'messages': [HumanMessage(content='My order ID is 1234567890', additional_kwargs={}, response_metadata={}, id='fd80b2aa-a139-4ba4-a29f-0a450c740aec'),
  AIMessage(content='Saved order 1234567890 for user user_a in thread a1', additional_kwargs={}, response_metadata={}, id='c9e3fe6e-0600-4fe9-9a47-d076af5b9917', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='d79c903a-e6b7-49da-ab86-9785be4089ba'),
  AIMessage(content='Your order 1234567890 is being processedIt will be shipped soon', additional_kwargs={}, response_metadata={}, id='cfe6e4fc-7dc4-49d2-9212-927e74e27cc4', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Remember my language is German', additional_kwargs={}, response_metadata={}, id='530cbd69-2966-489d-8678-6a7233b7d16b'),
  AIMessage(content='Saved your preferred language German', additional_kwargs={}, response_metadata={}, id='7c53cd5e-4011-4b7c-b545-1c676c153731', tool_call

In [71]:
chat("user_a", "a2", "Where is my order?", "premium")


User user_a | Thread a2

Input Where is my order?
[Node 1] thread_id: a2
[Node 1] message_count: 1
[Node 1] existing order: N/A
Answer:  Couldn't find your order. Please provide order ID
Total messages:  2


{'messages': [HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='5b98f4c7-5e3b-46eb-bbf7-17a9bfdc7b88'),
  AIMessage(content="Couldn't find your order. Please provide order ID", additional_kwargs={}, response_metadata={}, id='c5910068-6390-491a-9792-62ce01ef01a2', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'ask_order',
 'preferred_language': 'German'}

In [73]:
chat("user_a", "a2", "Which language do I prefer", "premium")


User user_a | Thread a2

Input Which language do I prefer
[Node 1] thread_id: a2
[Node 1] message_count: 5
[Node 1] existing order: N/A
Answer:  Your preferred language is German
Total messages:  6


{'messages': [HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='5b98f4c7-5e3b-46eb-bbf7-17a9bfdc7b88'),
  AIMessage(content="Couldn't find your order. Please provide order ID", additional_kwargs={}, response_metadata={}, id='c5910068-6390-491a-9792-62ce01ef01a2', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Which language do I prefer?', additional_kwargs={}, response_metadata={}, id='e1be8556-bfd6-4ae2-a8a7-0a203ebf7df2'),
  AIMessage(content='Ask about your order or language preferences.', additional_kwargs={}, response_metadata={}, id='7691cb81-aac5-4682-9cdf-0e11f8d99dff', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Which language do I prefer', additional_kwargs={}, response_metadata={}, id='bbda338f-cd21-41d0-8eaf-e8390ad7c34e'),
  AIMessage(content='Your preferred language is German', additional_kwargs={}, response_metadata={}, id='756e12b4-e94a-47ac-a508-c1028915f446', tool_calls=[], invalid_tool_calls=

In [74]:
chat("user_b", "b1", "Which language do I prefer", "premium")


User user_b | Thread b1

Input Which language do I prefer
[Node 1] thread_id: b1
[Node 1] message_count: 1
[Node 1] existing order: N/A
Answer:  Your preferred language is not set
Total messages:  2


{'messages': [HumanMessage(content='Which language do I prefer', additional_kwargs={}, response_metadata={}, id='8159a163-02a4-42cf-bec4-b5e4559d2741'),
  AIMessage(content='Your preferred language is not set', additional_kwargs={}, response_metadata={}, id='7fd05c1d-9c07-428b-b357-ae903bdb61d4', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'ask_language',
 'preferred_language': 'not set'}